# 패키지 및 데이터

In [25]:
origin = load_data("restaurant_sales_preprocessed")
origin.head()

어느 식당의 1년간 일별 매출을 기록한 데이터의 전처리 완료 버전(명목형이 이진변수만 있으므로 더미변수는 처리하지 않음)


,date,sales,visitors,avg_price,marketing_cost,delivery_ratio,rain_mm,temperature,holiday,weekend
0,2024-01-01,16.169,122,14490,9.200,0.480,1.946,5.200,0,0
1,2024-01-02,16.005,106,11880,9.149,0.480,2.079,11.100,0,0
2,2024-01-03,16.354,120,18010,8.666,0.430,2.595,12.900,1,0
3,2024-01-04,16.082,115,11160,8.748,0.410,2.028,12.900,0,0
4,2024-01-05,16.113,124,15480,8.594,0.510,2.197,11.000,0,0


### 카테고리 변환

In [26]:
origin = load_data("restaurant_sales_preprocessed")
origin.set_index("date", inplace=True)
origin["holiday"] = origin["holiday"].astype("int")
origin["weekend"] = origin["weekend"].astype("int")
origin.info()



어느 식당의 1년간 일별 매출을 기록한 데이터의 전처리 완료 버전(명목형이 이진변수만 있으므로 더미변수는 처리하지 않음)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 353 entries, 2024-01-01 to 2024-12-30
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sales           353 non-null    float64
 1   visitors        353 non-null    int64  
 2   avg_price       353 non-null    int64  
 3   marketing_cost  353 non-null    float64
 4   delivery_ratio  353 non-null    float64
 5   rain_mm         353 non-null    float64
 6   temperature     353 non-null    float64
 7   holiday         353 non-null    int32  
 8   weekend         353 non-null    int32  
dtypes: float64(5), int32(2), int64(2)
memory usage: 24.8 KB


### PyCaret setup

In [29]:
from pycaret.regression import RegressionExperiment

# ── Regression Experiment 생성
s = RegressionExperiment()

# ── 실험 환경 설정
s.setup(
    # ─────────────────────────────────
    # 기본 설정 (필수)
    # ─────────────────────────────────
    data=origin,                 # 데이터프레임
    target="sales",              # 예측 타깃
    session_id=52,               # 랜덤 시드 고정

    # ─────────────────────────────────
    # 데이터 분할
    # ─────────────────────────────────
    train_size=0.75,             # Train 비율
    fold=5,                      # CV fold 수

    # ─────────────────────────────────
    # 출력 / 실행 관련
    # ─────────────────────────────────
    verbose=False,               # 로그 출력 최소화
    

    # ─────────────────────────────────
    # 범주형 / 제외 변수
    # ─────────────────────────────────
    categorical_features=["weekend", "holiday"],
    ignore_features=[],

    # ─────────────────────────────────
    # 정규화 / 스케일링
    # ─────────────────────────────────
    normalize=True,
    normalize_method="zscore",   # 'minmax', 'maxabs', 'robust', 'zscore'

    # ─────────────────────────────────
    # 기타 전처리 옵션
    # ─────────────────────────────────
    remove_outliers=False,       # 이상치 제거
    transform_target=False,      # 타깃 변환
    feature_selection=False,     # 변수 선택 자동화
)


In [30]:
s.pull()

,Description,Value
0,Session id,52
1,Target,sales
2,Target type,Regression
3,Original data shape,"(353, 9)"
4,Transformed data shape,"(353, 9)"
5,Transformed train set shape,"(264, 9)"
6,Transformed test set shape,"(89, 9)"
7,Numeric features,6
8,Categorical features,2
9,Preprocess,True


# 베이스 모델 구축하기

### 사용할 수 있는 학습 모델의 종류 확인

In [35]:
s.models()

,Name,Reference,Turbo
ID,,,
lr,Linear Regression,sklearn.linear_model._base.LinearRegression,True
lasso,Lasso Regression,sklearn.linear_model._coordinate_descent.Lasso,True
ridge,Ridge Regression,sklearn.linear_model._ridge.Ridge,True
en,Elastic Net,sklearn.linear_model._coordinate_descent.ElasticNet,True
lar,Least Angle Regression,sklearn.linear_model._least_angle.Lars,True
llar,Lasso Least Angle Regression,sklearn.linear_model._least_angle.LassoLars,True
omp,Orthogonal Matching Pursuit,sklearn.linear_model._omp.OrthogonalMatchingPursuit,True
br,Bayesian Ridge,sklearn.linear_model._bayes.BayesianRidge,True
ard,Automatic Relevance Determination,sklearn.linear_model._bayes.ARDRegression,False


# 베이스 모델 성능 비교

### 모든 모델에 대한 성능 비교

In [36]:
best5models = s.compare_models(sort="RMSE", n_select=5, fold=5)
best5models

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
br,Bayesian Ridge,0.1654,0.0427,0.2056,0.6959,0.0119,0.0101,0.0320
ridge,Ridge Regression,0.1657,0.0428,0.2060,0.6948,0.0119,0.0101,0.3280
lr,Linear Regression,0.1658,0.0429,0.2061,0.6943,0.0119,0.0101,2.0920
lar,Least Angle Regression,0.1658,0.0429,0.2061,0.6943,0.0119,0.0101,0.0280
huber,Huber Regressor,0.1653,0.0429,0.2062,0.6938,0.0119,0.0101,0.0140
rf,Random Forest Regressor,0.1751,0.0494,0.2214,0.6438,0.0128,0.0107,0.0360
ada,AdaBoost Regressor,0.1758,0.0500,0.2227,0.6430,0.0128,0.0108,0.0240
et,Extra Trees Regressor,0.1783,0.0502,0.2231,0.6391,0.0129,0.0109,0.0320
lightgbm,Light Gradient Boosting Machine,0.1799,0.0517,0.2256,0.6329,0.0130,0.0110,0.0300
catboost,CatBoost Regressor,0.1851,0.0534,0.2299,0.6235,0.0132,0.0113,0.6180


[BayesianRidge(),
 Ridge(random_state=52),
 LinearRegression(n_jobs=-1),
 Lars(random_state=52),
 HuberRegressor()]

# 앙상블

### 선정된 모형에 대한 Voting

In [13]:
blended = s.blend_models(estimator_list=best5models, fold=5)
blended

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1826,0.0530,0.2302,0.6480,0.0132,0.0111
1,0.1683,0.0419,0.2046,0.7633,0.0118,0.0103
2,0.1438,0.0325,0.1802,0.7959,0.0105,0.0088
3,0.1538,0.0357,0.1889,0.6905,0.0109,0.0094
4,0.1790,0.0508,0.2255,0.5771,0.0130,0.0109
Mean,0.1655,0.0428,0.2059,0.6950,0.0119,0.0101
Std,0.0148,0.0081,0.0196,0.0787,0.0011,0.0009


VotingRegressor(estimators=[('Bayesian Ridge', BayesianRidge()),
                            ('Ridge Regression', Ridge(random_state=52)),
                            ('Linear Regression', LinearRegression(n_jobs=-1)),
                            ('Least Angle Regression', Lars(random_state=52)),
                            ('Huber Regressor', HuberRegressor())],
                n_jobs=-1)

In [ ]:
%%time

# ── 블렌딩 모델 하이퍼파라미터 튜닝
tuned = s.tune_model(
    estimator=blended,
    optimize="RMSE",

    # ── 탐색 설정
    n_iter=30,
    fold=3,
    choose_better=True,
    early_stopping=True,

    # ── 출력 설정
    verbose=False,

    # ── 탐색 방식
    # grid = 전수 탐색 / random = 무작위 탐색
    search_algorithm="grid",

    # ── 사용자 정의 하이퍼파라미터 그리드
    custom_grid={
        # ─────────────────────────────
        # Ridge Regression
        # ─────────────────────────────
        "Ridge Regression_alpha": [0.01, 0.1, 1, 10, 100],

        # ─────────────────────────────
        # Light Gradient Boosting Machine
        # ─────────────────────────────
        "Light Gradient Boosting Machine_n_estimators": [200, 500],
        "Light Gradient Boosting Machine_learning_rate": [0.05, 0.1],
        "Light Gradient Boosting Machine_num_leaves": [31, 63],
        "Light Gradient Boosting Machine_max_depth": [-1, 5],
        "Light Gradient Boosting Machine_min_child_samples": [20, 50],
        "Light Gradient Boosting Machine_subsample": [0.8, 1.0],
        "Light Gradient Boosting Machine_reg_alpha": [0, 0.1, 1],
        "Light Gradient Boosting Machine_reg_lambda": [0, 1, 5],

        # ─────────────────────────────
        # Support Vector Regression
        # ─────────────────────────────
        "Support Vector Regression_kernel": ["rbf"],          # 실습: rbf 고정
        "Support Vector Regression_C": [0.1, 1, 10, 100],
        "Support Vector Regression_epsilon": [0.01, 0.05, 0.1, 0.2],
        "Support Vector Regression_gamma": ["scale", "auto", 0.01, 0.1, 1],

        # ─────────────────────────────
        # CatBoost Regressor
        # ─────────────────────────────
        "CatBoost Regressor_iterations": [300, 500],
        "CatBoost Regressor_learning_rate": [0.01, 0.03, 0.1],
        "CatBoost Regressor_depth": [4, 6, 8],
        "CatBoost Regressor_l2_leaf_reg": [1, 3, 5],
        "CatBoost Regressor_subsample": [0.8, 1.0],
    },
)

tuned


### 성능평가

In [39]:
init_pyplot()

NameError: name 'init_pyplot' is not defined

### 성능평가 지표 확인

In [24]:
#%%time
hs_get_score_cv(tuned, X_train_transformed, y_train_transformed, X_transformed, y_transformed)

NameError: name 'hs_get_score_cv' is not defined